# Modelos finales (#20 → #21)

Entrena los modelos finales con **todos** los pares del K-fold (train + val, 159 pares), sin validación, durante la mediana de las mejores épocas de sus 5 folds (`resultados/kfold/<modelo>/resultado.json`). Son los que se evalúan en `test_anelo` y `test_externo` en el #21.

- Modelo 1 (siamesa, 6 bandas, fine-tuning): el elegido en `resultados/kfold/modelos.md`.
- Modelo 2 (fusión temprana): para comparar siamesa contra fusión en test (H4, y H3 en test externo).

Sube a `gs://vmseg-imagenes/modelos/final/<NOMBRE>/`: `final.pt` (pesos de la última época), `final.json` (configuración, media y desvío de normalización) y `curvas.png`.

Para GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU. Abrir Colab en incógnito con una sola cuenta (la de UdeSA). ~6 min por modelo.

In [ ]:
import os
import sys

try:
    import google.colab
    COLAB = True
except:
    COLAB = False

RAMA = "20-entrenamientos"
REPO = "/content/vacamuerta-metano-seg"

if COLAB:
    # si ya se clonó (por ejemplo, al correr la celda de nuevo), no se vuelve a clonar
    if not os.path.exists(REPO):
        !git clone -b {RAMA} https://github.com/sofia2005222/vacamuerta-metano-seg.git {REPO}
    %cd {REPO}
    !pip install -qq -e .
    # el notebook no ve el paquete recién instalado hasta reiniciar: agrego src/ a mano
    sys.path.insert(0, os.path.join(REPO, "src"))
else:
    %cd ..
    %load_ext autoreload
    %autoreload 2



In [ ]:
# bajar los pares del bucket
if COLAB:
    from google.colab import auth
    auth.authenticate_user()
    !mkdir -p data/pares
    !gsutil -m -q cp gs://vmseg-imagenes/pares/indice.csv "gs://vmseg-imagenes/pares/*_t1.tif" "gs://vmseg-imagenes/pares/*_t2.tif" "gs://vmseg-imagenes/pares/*_mask.tif" data/pares/


In [ ]:
import json
import numpy as np
import torch
from vmseg.config import DatasetConfig, TrainingConfig
from vmseg.kfold import entrenar_final

# los mismos experimentos del K-fold; las épocas salen de sus resultados
FINALES = [
    dict(nombre="modelo1_siamesa_6b_finetuning", arquitectura="siamesa", bandas=(0, 1, 2, 3, 4, 5), indices=False, modo="finetuning"),
    dict(nombre="modelo2_fusion_6b_finetuning",  arquitectura="fusion",  bandas=(0, 1, 2, 3, 4, 5), indices=False, modo="finetuning"),
]
PERDIDA = "ce_dice"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

for exp in FINALES:
    kfold = json.load(open(f"resultados/kfold/{exp['nombre']}/resultado.json"))
    exp["epocas"] = int(np.median(kfold["mejores_epocas"]))
    exp["iou_kfold"] = (kfold["iou_val_media"], kfold["iou_val_desvio"])
    print(exp["nombre"], "| mejores épocas", kfold["mejores_epocas"], "→", exp["epocas"], "épocas")

In [ ]:
def entrenar_y_subir(exp):
    print(f"\n#################### {exp['nombre']} ({exp['epocas']} épocas) ####################")
    carpeta = f"runs/final/{exp['nombre']}"
    dataset_config = DatasetConfig(data_root="data/pares", bandas=exp["bandas"], indices=exp["indices"],
                                   num_classes=2, batch_size=8, augment=True)
    train_config = TrainingConfig(epochs=exp["epocas"], learning_rate=1e-3, device=device,
                                  checkpoints_dir=carpeta, perdida=PERDIDA)

    history, mean, std = entrenar_final(dataset_config, train_config, exp["arquitectura"], exp["modo"])

    # todo lo que hace falta para cargar el modelo y normalizar los pares de test igual que en train
    info = dict(nombre=exp["nombre"], arquitectura=exp["arquitectura"], bandas=list(exp["bandas"]),
                indices=exp["indices"], modo=exp["modo"], num_classes=2, perdida=PERDIDA,
                epocas=exp["epocas"], lr=train_config.learning_rate, pares_train=history["pares_train"],
                iou_train_ultima=round(history["train_miou"][-1], 4),
                iou_kfold_media=exp["iou_kfold"][0], iou_kfold_desvio=exp["iou_kfold"][1],
                mean=mean.tolist(), std=std.tolist())
    with open(f"{carpeta}/final.json", "w") as f:
        json.dump(info, f, indent=2)

    if COLAB:
        destino = f"gs://vmseg-imagenes/modelos/final/{exp['nombre']}/"
        codigo = os.system(f"gsutil cp {carpeta}/final.pt {carpeta}/final.json {carpeta}/curvas.png {destino}")
        print("subido" if codigo == 0 else "NO se pudo subir", destino)
    return info

infos = [entrenar_y_subir(exp) for exp in FINALES]
for info in infos:
    print(f"{info['nombre']}: {info['epocas']} épocas, {info['pares_train']} pares, IoU train última época {info['iou_train_ultima']:.3f}")

## Cómo cargar un modelo final (para #21, #25, #26)

```python
import json, torch
from vmseg.config import DatasetConfig
from vmseg.dataset_pares import ParesDataset
from vmseg.siamesa import crear_modelo, nombres_canales

info = json.load(open("final.json"))
config = DatasetConfig(data_root="data/pares", bandas=tuple(info["bandas"]), indices=info["indices"],
                       num_classes=info["num_classes"], batch_size=8)
model = crear_modelo(info["arquitectura"], nombres_canales(config), num_classes=info["num_classes"], modo=info["modo"])
model.load_state_dict(torch.load("final.pt", map_location="cpu"))
model.eval()

# los pares de test se normalizan con la media y el desvío del entrenamiento (no con los del test)
mean, std = torch.tensor(info["mean"]), torch.tensor(info["std"])
test = ParesDataset(config, "test_anelo", mean=mean, std=std)
```